# Can we predict 30-day readmission?
Target: `readmit_30d`. Each row is one hospital stay; people appear on several rows (~8 each), so all evaluation is **grouped by `person_key`**. Otherwise the same person lands in both train and test and scores are inflated.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedGroupKFold, KFold, cross_val_predict, GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
pd.set_option('display.width',140)
df=pd.read_csv('data.csv'); print(pd.read_csv('columns.csv').to_string(index=False)); df.shape

## 1. Data checks

In [ ]:
print(df.isna().sum().sum(),'missing;',df.duplicated().sum(),'duplicate rows;',df.person_key.nunique(),'people')
p=df.readmit_30d.mean(); print('prevalence:',round(p,4))
df.describe().T

In [ ]:
# Are person-level attributes constant within person? Is the outcome clustered by person?
g=df.groupby('person_key')
print('people with varying value:',{c:int((g[c].nunique()>1).sum()) for c in ['birth_year','sex','postcode_area']})
pr=g.readmit_30d.agg(['mean','size'])
exp_var=(p*(1-p)/pr['size']).mean()
print(f'variance of person readmit rates: observed {pr["mean"].var():.4f} vs expected if independent {exp_var:.4f}')

In [ ]:
num=['birth_year','postcode_area','bmi','length_of_stay_days','n_medications','lab_a','lab_b','visit_number']
fig,ax=plt.subplots(2,4,figsize=(16,7))
for a,c in zip(ax.flat,num):
    q=pd.qcut(df[c],8,duplicates='drop'); r=df.groupby(q,observed=True).readmit_30d.mean()
    a.plot(range(len(r)),r.values,'o-'); a.axhline(p,ls='--',c='gray'); a.set_title(c); a.set_xlabel('octile'); a.set_ylabel('readmit rate')
plt.tight_layout(); plt.show()
print(df.groupby('sex').readmit_30d.mean())

## 2. Models, evaluated with grouped 5-fold CV
`person_key` is an identifier and is not used as a feature. `postcode_area` is a code, so it is one-hot encoded for the logistic model.

In [ ]:
X=df.drop(columns=['person_key','readmit_30d']); X['male']=(X.sex=='M').astype(int); X=X.drop(columns='sex')
y=df.readmit_30d; groups=df.person_key
cont=[c for c in X if c!='postcode_area']
lr=make_pipeline(ColumnTransformer([('n',StandardScaler(),cont),('p',OneHotEncoder(handle_unknown='ignore'),['postcode_area'])]),LogisticRegression(max_iter=2000))
models={'baseline (prevalence)':DummyClassifier(strategy='prior'),'logistic regression':lr,
        'gradient boosting':HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,l2_regularization=1.0,random_state=0)}
cv=StratifiedGroupKFold(5,shuffle=True,random_state=0)
oof={}; rows=[]
for n,m in models.items():
    oof[n]=cross_val_predict(m,X,y,groups=groups,cv=cv,method='predict_proba')[:,1]
    fa=[roc_auc_score(y.iloc[t],oof[n][t]) for _,t in cv.split(X,y,groups)]
    rows.append((n,roc_auc_score(y,oof[n]),np.std(fa),average_precision_score(y,oof[n]),brier_score_loss(y,oof[n])))
res=pd.DataFrame(rows,columns=['model','ROC AUC','AUC sd (folds)','PR AUC','Brier']).set_index('model'); res.round(4)

### Is the signal real? Person-level bootstrap CI and label-shuffle null

In [ ]:
rng=np.random.default_rng(0)
best=res.drop('baseline (prevalence)')['ROC AUC'].idxmax()
pp=oof[best]; people=df.person_key.unique(); idx=df.groupby('person_key').indices
boot=[]
for _ in range(500):
    s=rng.choice(people,len(people)); ii=np.concatenate([idx[k] for k in s]); boot.append(roc_auc_score(y.values[ii],pp[ii]))
print(best,'AUC 95% CI (person bootstrap):',np.percentile(boot,[2.5,97.5]).round(3))
null=[]
for i in range(20):
    ys=pd.Series(rng.permutation(y.values))
    o=cross_val_predict(models[best],X,ys,groups=groups,cv=cv,method='predict_proba')[:,1]; null.append(roc_auc_score(ys,o))
print('label-shuffled AUC: mean %.3f, max %.3f (20 runs)'%(np.mean(null),np.max(null)))

### Leakage demonstration: ignoring person grouping

In [ ]:
o=cross_val_predict(models['gradient boosting'],X,y,cv=KFold(5,shuffle=True,random_state=0),method='predict_proba')[:,1]
print('Row-wise KFold AUC (leaky): %.3f   vs grouped: %.3f'%(roc_auc_score(y,o),res.loc['gradient boosting','ROC AUC']))

## 3. Feature importance and calibration

In [ ]:
tr,te=next(GroupShuffleSplit(1,test_size=0.25,random_state=1).split(X,y,groups))
m=models[best].fit(X.iloc[tr],y.iloc[tr])
pi=permutation_importance(m,X.iloc[te],y.iloc[te],scoring='roc_auc',n_repeats=20,random_state=0)
imp=pd.Series(pi.importances_mean,X.columns).sort_values(); err=pd.Series(pi.importances_std,X.columns)[imp.index]
fig,ax=plt.subplots(1,2,figsize=(14,5))
ax[0].barh(imp.index,imp.values,xerr=err.values); ax[0].set_title('Permutation importance (drop in held-out AUC)')
fo,mp=calibration_curve(y,oof[best],n_bins=10,strategy='quantile')
ax[1].plot(mp,fo,'o-'); ax[1].plot([0,.5],[0,.5],'--',c='gray'); ax[1].set_xlabel('predicted'); ax[1].set_ylabel('observed'); ax[1].set_title('Calibration (out-of-fold)')
plt.tight_layout(); plt.show()

In [ ]:
d=pd.DataFrame({'p':oof[best],'y':y}); d['decile']=pd.qcut(d.p,10,labels=False)+1
t=d.groupby('decile').y.agg(['mean','size']); t.columns=['observed rate','n']; t.round(3).T

## 4. Extension: prior readmission history
Readmissions cluster by person far beyond chance (section 1), so a person's earlier outcomes may help. Features below use **only earlier visits** (ordered by `visit_number`, shifted by one), so they are known at prediction time. Caveat: this needs the person's earlier visits to be in the system, and the first visit has no history (NaN, handled natively by gradient boosting).

In [ ]:
h=df.sort_values(['person_key','visit_number']).copy()
gg=h.groupby('person_key').readmit_30d
h['prior_readmits']=gg.cumsum()-h.readmit_30d
h['prior_visits']=gg.cumcount()
h['prior_rate']=h.prior_readmits/h.prior_visits.replace(0,np.nan)
h=h.sort_index()
X2=X.assign(prior_readmits=h.prior_readmits,prior_rate=h.prior_rate)
gb=models['gradient boosting']
o2=cross_val_predict(gb,X2,y,groups=groups,cv=cv,method='predict_proba')[:,1]
print('AUC without history: %.3f | with history: %.3f | PR AUC %.3f -> %.3f'%(
    roc_auc_score(y,oof['gradient boosting']),roc_auc_score(y,o2),
    average_precision_score(y,oof['gradient boosting']),average_precision_score(y,o2)))
v1=(df.visit_number==1).values
print('AUC on first-ever visits only (no history available): %.3f'%roc_auc_score(y[v1],o2[v1]))

## 5. Summary

In [ ]:
print(res.round(3).to_string())
print(f'\nTop-decile readmit rate {d[d.decile==10].y.mean():.1%} vs bottom decile {d[d.decile==1].y.mean():.1%} (overall {p:.1%})')